# 1484. Group Sold Products By The Date

## Problem Description

A sales log records which product was sold on a particular date. Repeated records can occur.

### Table Schema

**Activities**

| Column | Type | Meaning / constraint |
| --- | --- | --- |
| sell_date | date | date of sale |
| product | varchar | product name |

### Task

For each sell_date, calculate num_sold (the number of distinct products) and products (the distinct product names sorted lexicographically and joined by commas, without extra spaces). Return sell_date, num_sold, products sorted ascending by sell_date.

## Example

### Input

**activities**

| sell_date | product |
| --- | --- |
| 2024-01-01 | B |
| 2024-01-01 | A |
| 2024-01-01 | A |
| 2024-01-02 | C |

### Expected Output

| sell_date | num_sold | products |
| --- | --- | --- |
| 2024-01-01 | 2 | A,B |
| 2024-01-02 | 1 | C |

### Explanation

On January 1, A and B were sold; the repeated A does not increase the distinct count. The sorted product string is A,B. January 2 has only C.

## Practice

The next code cell constructs exactly these example DataFrames. Try the blank practice cell before reading the solutions.

## References

The problem is restated for self-contained practice; the examples are original. Community links are for further reading, and implementations below are independently written.

[Doocs problem reference](https://github.com/doocs/leetcode/blob/main/solution/1400-1499/1484.Group%20Sold%20Products%20By%20The%20Date/README_EN.md) · [LeetCode](https://leetcode.com/problems/group-sold-products-by-the-date/)

[Community solution index (Silversaya, 2023)](https://leetcode.cn/discuss/post/3573106/zhe-shi-pandas30tian-ji-hua-de-ti-jie-qu-v0wv/) · [LeetCode discussions](https://leetcode.com/problems/group-sold-products-by-the-date/solutions/)

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal

activities = pd.DataFrame([["2024-01-01","B"],["2024-01-01","A"],["2024-01-01","A"],["2024-01-02","C"]], columns=["sell_date","product"])
expected = pd.DataFrame([["2024-01-01",2,"A,B"],["2024-01-02",1,"C"]], columns=["sell_date","num_sold","products"])
display(activities)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions

### Solution 1 — Named aggregation
每個日期內的產品需去重及排序。

In [ ]:
def solution_1(activities):
    return activities.groupby('sell_date',as_index=False).agg(num_sold=('product','nunique'),products=('product',lambda s: ','.join(sorted(s.unique())))).sort_values('sell_date')

result_1 = solution_1(activities.copy())
display(result_1)

### Solution 2 — Deduplicate and sort before aggregation
先統一資料粒度，再使用 size 及 join。

In [ ]:
def solution_2(activities):
    d = activities.drop_duplicates(['sell_date','product']).sort_values(['sell_date','product'])
    return d.groupby('sell_date',as_index=False).agg(num_sold=('product','size'),products=('product',','.join)).sort_values('sell_date')

result_2 = solution_2(activities.copy())
display(result_2)

### Solution 3 — Aggregate list once
清單聚合後同時衍生兩個輸出欄位。

In [ ]:
def solution_3(activities):
    g = activities.groupby('sell_date').product.agg(lambda s: sorted(s.unique())).reset_index()
    return g.assign(num_sold=g.product.str.len(),products=g.product.str.join(','))[['sell_date','num_sold','products']].sort_values('sell_date')

result_3 = solution_3(activities.copy())
display(result_3)

## Check all solutions
Rows are normalized only for the example comparison; preserve required ordering in the solutions themselves.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Compare row grain, missing matches, duplicate keys and zero-count categories.